# Transformer Model Training for WESAD Dataset (64Hz Sampling Rate)

This notebook trains a Transformer model to classify stress, amusement, and baseline states using the WESAD dataset.

In [1]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.signal import resample_poly, find_peaks, butter, filtfilt
import glob
import math
import pandas as pd

## 1. Configuration and Setup

In [2]:
# Configuration
DATASET_PATH = '/home/binghin2/Myproject/Dataset/WESAD'
TRAIN_USERS = ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14']
VAL_USERS = ['S13', 'S14'] 
TEST_USERS = ['S15', 'S16', 'S17']
TARGET_LABELS = {1: 0, 2: 1, 3: 2} # 1: baseline, 2: stress, 3: amusement -> 0, 1, 2
N_FEATURES = 10 # ACC (3), BVP (1), SCL (1), SCR (1), TEMP (1), HR (1), IBI (1), HRV (1)
DOWNSAMPLE_RATE = 64 # Hz, the rate to downsample all signals to
WINDOW_SIZE_SEC = 60 # seconds # 0~60, 1~61 , 2~62 
TRAIN_STRIDE_SEC = 10 # default seconds for training (used when dynamic stride is disabled)
TEST_STRIDE_SEC = 60 # seconds (no overlap for evaluation)
LABEL_PURITY_THRESHOLD = 0.8 # Keep only windows where dominant label ratio >= 80%

# Dynamic stride for training windows (raw labels: 1=baseline, 2=stress, 3=amusement)
TRAIN_DYNAMIC_STRIDE_MAP = {
    1: 15,  # baseline -> sparse sampling
    2: 10,  # stress -> normal sampling
    3: 5    # amusement -> dense sampling (oversampling)
}

# Transformer Hyperparameters
D_MODEL = 32 # Embedding dimension
NHEAD = 4 # Number of attention heads
DIM_FEEDFORWARD = 128 # Dimension of feedforward network
NLAYERS = 2 # Number of transformer layers
OUTPUT_DIM = len(TARGET_LABELS)
BATCH_SIZE = 64
NUM_EPOCHS = 50
LEARNING_RATE = 0.0005
DROPOUT = 0.5

# Setup device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda:0


## 2. Data Loading and Preprocessing

In [3]:
def _to_2d_column(signal):
    """Ensures signal is 2D: (time, channels)."""
    signal = np.asarray(signal)
    if signal.ndim == 1:
        return signal.reshape(-1, 1)
    return signal

def _resample_signal(signal, orig_rate, target_rate):
    """Anti-aliasing resampling using polyphase filtering."""
    signal_2d = _to_2d_column(signal)
    if orig_rate == target_rate:
        return signal_2d
    return resample_poly(signal_2d, up=target_rate, down=orig_rate, axis=0)

def decompose_eda_scl_scr(eda, fs=4, cutoff_hz=0.05, order=2):
    """Decompose EDA into SCL (tonic) and SCR (phasic)."""
    eda_flat = np.asarray(eda).reshape(-1).astype(float)
    nyq = 0.5 * fs
    normalized_cutoff = cutoff_hz / nyq
    b, a = butter(order, normalized_cutoff, btype='low')

    if len(eda_flat) <= (max(len(a), len(b)) * 3):
        scl = np.copy(eda_flat)
    else:
        scl = filtfilt(b, a, eda_flat)

    scr = eda_flat - scl
    return scl.reshape(-1, 1), scr.reshape(-1, 1)

def extract_bvp_features(bvp, orig_rate=64, target_rate=64):
    """
    Extract HR, IBI, and HRV from raw BVP and interpolate them to target_rate.
    Returns shape: (N, 3) -> [HR, IBI, HRV]
    """
    bvp_flat = np.asarray(bvp).flatten()

    min_distance = int(orig_rate * 0.3)
    peaks, _ = find_peaks(bvp_flat, distance=min_distance)

    times_orig = np.arange(len(bvp_flat)) / float(orig_rate)
    target_len = int(np.ceil(len(bvp_flat) * target_rate / float(orig_rate)))
    times_target = np.arange(target_len) / float(target_rate)

    if len(peaks) > 1:
        peak_times = times_orig[peaks]
        ibi = np.diff(peak_times)
        hr = 60.0 / np.clip(ibi, 1e-6, None)
        peak_times = peak_times[1:]

        hr_continuous = np.interp(times_target, peak_times, hr)
        ibi_continuous = np.interp(times_target, peak_times, ibi)

        ibi_series = pd.Series(ibi_continuous)
        hrv_continuous = (
            ibi_series
            .rolling(window=target_rate * 10, min_periods=1)
            .std()
            .fillna(0.0)
            .values
        )
    else:
        hr_continuous = np.zeros(target_len)
        ibi_continuous = np.zeros(target_len)
        hrv_continuous = np.zeros(target_len)

    hr_continuous = hr_continuous.reshape(-1, 1)
    ibi_continuous = ibi_continuous.reshape(-1, 1)
    hrv_continuous = hrv_continuous.reshape(-1, 1)

    return np.concatenate([hr_continuous, ibi_continuous, hrv_continuous], axis=1)

def load_and_preprocess_data(subject_path):
    """Loads data, extracts BVP/EDA features, resamples, synchronizes, and normalizes."""
    with open(subject_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    # --- Extract Wrist Data --- #
    wrist_data = data['signal']['wrist']
    acc = wrist_data['ACC']      # 32 Hz, shape: (T, 3)
    bvp = wrist_data['BVP']      # 64 Hz, shape: (T,) or (T, 1)
    eda = wrist_data['EDA']      # 4 Hz,  shape: (T,) or (T, 1)
    temp = wrist_data['TEMP']    # 4 Hz,  shape: (T,) or (T, 1)
    labels = data['label']       # 700 Hz

    # Extract HR/IBI/HRV from raw 64 Hz BVP before downsampling
    bvp_features = extract_bvp_features(bvp, orig_rate=64, target_rate=DOWNSAMPLE_RATE)

    # Decompose EDA(4Hz) -> SCL/SCR before downsampling
    scl_eda, scr_eda = decompose_eda_scl_scr(eda, fs=4, cutoff_hz=0.05, order=2)

    # --- Resample raw signals to DOWNSAMPLE_RATE --- #
    acc_down = _resample_signal(acc, orig_rate=32, target_rate=DOWNSAMPLE_RATE)
    bvp_down = _resample_signal(bvp, orig_rate=64, target_rate=DOWNSAMPLE_RATE)
    scl_down = _resample_signal(scl_eda, orig_rate=4, target_rate=DOWNSAMPLE_RATE)
    scr_down = _resample_signal(scr_eda, orig_rate=4, target_rate=DOWNSAMPLE_RATE)
    temp_down = _resample_signal(temp, orig_rate=4, target_rate=DOWNSAMPLE_RATE)

    # Align labels by nearest timestamp (categorical label -> no interpolation)
    label_timestamps = np.arange(len(labels)) / 700.0
    data_timestamps = np.arange(len(acc_down)) / float(DOWNSAMPLE_RATE)
    idx = np.searchsorted(label_timestamps, data_timestamps, side='left')
    idx = np.clip(idx, 0, len(labels) - 1)
    labels_down = labels[idx].astype(int)

    # Find minimum common length and truncate
    min_len = min(
        len(acc_down), len(bvp_down), len(scl_down), len(scr_down), len(temp_down), len(bvp_features), len(labels_down)
    )

    # --- Combine Features --- #
    features = np.concatenate([
        acc_down[:min_len],
        bvp_down[:min_len],
        scl_down[:min_len],
        scr_down[:min_len],
        temp_down[:min_len],
        bvp_features[:min_len]
    ], axis=1)

    # --- Subject-wise normalization --- #
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    labels_final = labels_down[:min_len]

    return features_scaled, labels_final

def create_windows(features, labels, stride_sec, dynamic_stride_map=None, apply_dynamic_stride=False):
    """Creates sliding windows and keeps only label-pure windows.

    - Fixed mode: uses stride_sec for all windows.
    - Dynamic mode: stride is selected by dominant raw label in each window.
    """
    window_samples = WINDOW_SIZE_SEC * DOWNSAMPLE_RATE
    default_stride_samples = max(1, int(stride_sec * DOWNSAMPLE_RATE))

    X, y = [], []
    i = 0
    max_start = len(features) - window_samples

    while i <= max_start:
        window_features = features[i : i + window_samples]
        window_labels = labels[i : i + window_samples].astype(int)

        counts = np.bincount(window_labels)
        if counts.size == 0:
            i += default_stride_samples
            continue

        most_frequent_label = int(counts.argmax())
        label_purity = counts[most_frequent_label] / float(window_samples)

        if most_frequent_label in TARGET_LABELS and label_purity >= LABEL_PURITY_THRESHOLD:
            X.append(window_features)
            y.append(TARGET_LABELS[most_frequent_label])

        if apply_dynamic_stride and dynamic_stride_map is not None:
            stride_next_sec = dynamic_stride_map.get(most_frequent_label, stride_sec)
            stride_next_samples = max(1, int(stride_next_sec * DOWNSAMPLE_RATE))
        else:
            stride_next_samples = default_stride_samples

        i += stride_next_samples

    return np.array(X), np.array(y)

In [4]:
# --- Process TRAIN users (overlap allowed). Skip users reserved for VAL/TEST --- #
train_X, train_y = [], []
for user in TRAIN_USERS:
    if user in VAL_USERS or user in TEST_USERS:
        print(f'Skipping user {user} (reserved for VAL/TEST)')
        continue
    subject_path = os.path.join(DATASET_PATH, user, f'{user}.pkl')
    if os.path.exists(subject_path):
        print(f'Processing TRAIN user: {user}...')
        features, labels = load_and_preprocess_data(subject_path)
        X_user, y_user = create_windows(
            features,
            labels,
            stride_sec=TRAIN_STRIDE_SEC,
            dynamic_stride_map=TRAIN_DYNAMIC_STRIDE_MAP,
            apply_dynamic_stride=True
        )
        if X_user.size > 0:
            train_X.append(X_user)
            train_y.append(y_user)

if len(train_X) == 0:
    raise ValueError('No training data found. Check TRAIN_USERS or DATASET_PATH.')

X_train = np.concatenate(train_X, axis=0)
y_train = np.concatenate(train_y, axis=0)

print(f'\nTotal TRAIN windows created: {len(X_train)}')
print(f'TRAIN feature shape: {X_train.shape}')
print(f'TRAIN label distribution: {np.bincount(y_train)}')

# --- Process VAL users (no overlap by default using TEST_STRIDE_SEC) --- #
val_X, val_y = [], []
for user in VAL_USERS:
    subject_path = os.path.join(DATASET_PATH, user, f'{user}.pkl')
    if os.path.exists(subject_path):
        print(f'Processing VAL user: {user}...')
        features, labels = load_and_preprocess_data(subject_path)
        X_user, y_user = create_windows(features, labels, stride_sec=TEST_STRIDE_SEC)
        if X_user.size > 0:
            val_X.append(X_user)
            val_y.append(y_user)

if len(val_X) == 0:
    raise ValueError('No validation data found. Check VAL_USERS or DATASET_PATH.')

X_val = np.concatenate(val_X, axis=0)
y_val = np.concatenate(val_y, axis=0)

print(f'\nTotal VAL windows created: {len(X_val)}')
print(f'VAL feature shape: {X_val.shape}')
print(f'VAL label distribution: {np.bincount(y_val)}')

# --- Process TEST users (non-overlapping windows) --- #
test_X, test_y = [], []
for user in TEST_USERS:
    subject_path = os.path.join(DATASET_PATH, user, f'{user}.pkl')
    if os.path.exists(subject_path):
        print(f'Processing TEST user: {user}...')
        features, labels = load_and_preprocess_data(subject_path)
        X_user, y_user = create_windows(features, labels, stride_sec=TEST_STRIDE_SEC)
        if X_user.size > 0:
            test_X.append(X_user)
            test_y.append(y_user)

if len(test_X) == 0:
    raise ValueError('No test data found. Check TEST_USERS or DATASET_PATH.')

X_test = np.concatenate(test_X, axis=0)
y_test = np.concatenate(test_y, axis=0)

print(f'\nTest set size: {len(X_test)}')
print(f'TEST feature shape: {X_test.shape}')
print(f'TEST label distribution: {np.bincount(y_test)}')

# Subject-wise normalization is already applied in load_and_preprocess_data()

Processing TRAIN user: S2...
Processing TRAIN user: S3...
Processing TRAIN user: S4...
Processing TRAIN user: S5...
Processing TRAIN user: S6...
Processing TRAIN user: S7...
Processing TRAIN user: S8...
Processing TRAIN user: S9...
Processing TRAIN user: S10...
Processing TRAIN user: S11...
Skipping user S13 (reserved for VAL/TEST)
Skipping user S14 (reserved for VAL/TEST)

Total TRAIN windows created: 2041
TRAIN feature shape: (2041, 3840, 10)
TRAIN label distribution: [758 616 667]
Processing VAL user: S13...
Processing VAL user: S14...

Total VAL windows created: 71
VAL feature shape: (71, 3840, 10)
VAL label distribution: [38 21 12]
Processing TEST user: S15...
Processing TEST user: S16...
Processing TEST user: S17...

Test set size: 108
TEST feature shape: (108, 3840, 10)
TEST label distribution: [57 33 18]


## 3. PyTorch Dataset and DataLoader

In [5]:
class WesadDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = WesadDataset(X_train, y_train)
val_dataset = WesadDataset(X_val, y_val)
test_dataset = WesadDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 4. Transformer Model Definition

In [6]:
class PositionalEncoding(nn.Module):
    """Positional encoding for Transformer model (from PyTorch docs)"""
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        return x + self.pe[:, :x.size(1), :]

class TransformerModel(nn.Module):
    def __init__(self, input_dim, d_model, nhead, dim_feedforward, nlayers, output_dim, dropout=0.1):
        super(TransformerModel, self).__init__()
        
        # Input projection layer (project input features to d_model dimension)
        self.input_proj = nn.Linear(input_dim, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Transformer encoder layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        # Stack multiple transformer encoder layers
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=nlayers)
        
        # Output layer
        self.fc = nn.Linear(d_model, output_dim)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        
        # Project input to d_model dimension
        x = self.input_proj(x)  # (batch_size, seq_len, d_model)
        
        # Add positional encoding
        x = self.pos_encoder(x)  # (batch_size, seq_len, d_model)
        
        # Transformer encoder
        x = self.transformer_encoder(x)  # (batch_size, seq_len, d_model)
        
        # Use the last token's output for classification
        x = x[:, -1, :]  # (batch_size, d_model)
        
        # Output layer
        x = self.fc(x)  # (batch_size, output_dim)
        
        return x

## 5. Model Training

In [7]:
model = TransformerModel(
    input_dim=N_FEATURES,
    d_model=D_MODEL,
    nhead=NHEAD,
    dim_feedforward=DIM_FEEDFORWARD,
    nlayers=NLAYERS,
    output_dim=OUTPUT_DIM,
    dropout=DROPOUT
)
model.to(device)

# Weighted Cross-Entropy for class imbalance
class_counts = np.bincount(y_train, minlength=OUTPUT_DIM).astype(np.float32)
class_counts = np.clip(class_counts, a_min=1.0, a_max=None)
class_weights_np = 1.0 / class_counts
class_weights_np = class_weights_np / class_weights_np.sum()
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)
print(f"Class counts (train): {class_counts.astype(int)}")
print(f"Class weights       : {class_weights_np}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Early Stopping 설정
PATIENCE = 10
MIN_DELTA = 1e-4
best_val_loss = float("inf")
epochs_no_improve = 0
best_model_state = None

print("Starting training with Early Stopping...")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for i, (sequences, labels) in enumerate(train_loader):
        sequences = sequences.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * sequences.size(0)

        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_acc = (correct_predictions / total_samples) * 100

    # Validation
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for sequences, labels in val_loader:
            sequences = sequences.to(device)
            labels = labels.to(device)

            outputs = model(sequences)
            val_loss = criterion(outputs, labels)

            val_running_loss += val_loss.item() * sequences.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / val_total
    val_epoch_acc = (val_correct / val_total) * 100

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}% | "
        f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.2f}%"
    )

    # Early Stopping 체크 (Validation Loss 기준)
    if val_epoch_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_epoch_loss
        best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        print(f"  ✅ Validation loss improved. Best Val Loss: {best_val_loss:.4f}")
    else:
        epochs_no_improve += 1
        print(f"  ⏳ No improvement for {epochs_no_improve}/{PATIENCE} epoch(s)")

        if epochs_no_improve >= PATIENCE:
            print(f"\n🛑 Early stopping triggered at epoch {epoch+1}")
            break

# best weight 복원
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model.to(device)
    print(f"✅ Loaded best model weights (Best Val Loss: {best_val_loss:.4f})")

Class counts (train): [758 616 667]
Class weights       : [0.29700458 0.3654699  0.33752546]
Starting training with Early Stopping...
Epoch [1/50] | Train Loss: 0.8745, Train Acc: 61.49% | Val Loss: 0.7656, Val Acc: 50.70%
  ✅ Validation loss improved. Best Val Loss: 0.7656
Epoch [2/50] | Train Loss: 0.5614, Train Acc: 76.53% | Val Loss: 0.7188, Val Acc: 56.34%
  ✅ Validation loss improved. Best Val Loss: 0.7188
Epoch [3/50] | Train Loss: 0.4138, Train Acc: 83.39% | Val Loss: 0.8020, Val Acc: 61.97%
  ⏳ No improvement for 1/10 epoch(s)
Epoch [4/50] | Train Loss: 0.3501, Train Acc: 85.55% | Val Loss: 1.1200, Val Acc: 60.56%
  ⏳ No improvement for 2/10 epoch(s)
Epoch [5/50] | Train Loss: 0.3144, Train Acc: 87.11% | Val Loss: 1.1723, Val Acc: 57.75%
  ⏳ No improvement for 3/10 epoch(s)
Epoch [6/50] | Train Loss: 0.2983, Train Acc: 87.70% | Val Loss: 1.2232, Val Acc: 57.75%
  ⏳ No improvement for 4/10 epoch(s)
Epoch [7/50] | Train Loss: 0.2779, Train Acc: 88.68% | Val Loss: 1.2645, Val Acc

In [8]:
# 학습 완료 후 모델 저장
SAVE_DIR = r"C:\Users\mbnv6\Project\Research\WESAD_classification\Training\Save_model\Transformer"

os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_SAVE_PATH = os.path.join(SAVE_DIR, "Transformer_64hz_model.pt")

checkpoint = {
    "model_name": "Transformer_64HZ",
    "model_state_dict": model.state_dict(),
    "n_features": N_FEATURES,
    "d_model": D_MODEL,
    "nhead": NHEAD,
    "dim_feedforward": DIM_FEEDFORWARD,
    "nlayers": NLAYERS,
    "output_dim": OUTPUT_DIM,
}

torch.save(checkpoint, MODEL_SAVE_PATH)
print(f"✅ Model saved to: {MODEL_SAVE_PATH}")

✅ Model saved to: C:\Users\mbnv6\Project\Research\WESAD_classification\Training\Save_model\Transformer/Transformer_64hz_model.pt


## 6. Model Evaluation

In [9]:
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        outputs = model(sequences)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = 100 * correct / total
print(f'\nTest Accuracy: {accuracy:.2f} %')

# Optional: Print classification report for more details
try:
    from sklearn.metrics import classification_report
    report = classification_report(all_labels, all_preds, target_names=['baseline', 'stress', 'amusement'])
    print("\nClassification Report:")
    print(report)
except ImportError:
    print("\nPlease install scikit-learn to see the classification report: pip install -U scikit-learn")

# 각 클래스별 정확도 계산
all_labels_np = np.array(all_labels)
all_preds_np = np.array(all_preds)

class_names = ['Baseline', 'Stress', 'Amusement']
print("\n" + "="*60)
print("각 상태별 정확도")
print("="*60)

for class_idx, class_name in enumerate(class_names):
    # 해당 클래스의 샘플들
    class_mask = all_labels_np == class_idx
    class_total = class_mask.sum()
    
    if class_total > 0:
        # 해당 클래스에서 올바르게 예측된 샘플들
        class_correct = ((all_labels_np == class_idx) & (all_preds_np == class_idx)).sum()
        class_accuracy = (class_correct / class_total) * 100
        print(f"{class_name}: {class_accuracy:.2f}% ({class_correct}/{class_total})")
    else:
        print(f"{class_name}: N/A (no samples)")

print("="*60)


Test Accuracy: 49.07 %

Classification Report:
              precision    recall  f1-score   support

    baseline       0.67      0.32      0.43        57
      stress       0.76      0.79      0.78        33
   amusement       0.19      0.50      0.28        18

    accuracy                           0.49       108
   macro avg       0.54      0.53      0.49       108
weighted avg       0.62      0.49      0.51       108


각 상태별 정확도
Baseline: 31.58% (18/57)
Stress: 78.79% (26/33)
Amusement: 50.00% (9/18)


In [10]:
# Inference Time Benchmark (Transformer)
import time
import numpy as np
import torch

if "model" not in globals():
    raise NameError("`model` 변수가 없습니다. 모델 생성/학습 셀을 먼저 실행하세요.")
if "test_loader" not in globals():
    raise NameError("`test_loader` 변수가 없습니다. DataLoader 생성 셀을 먼저 실행하세요.")

runtime_device = device if "device" in globals() else torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(runtime_device)
model.eval()


def _sync_if_cuda(dev):
    if dev.type == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def benchmark_inference(model, data_loader, dev, warmup_steps=20):
    iterator = iter(data_loader)
    first_batch = next(iterator)
    x0 = first_batch[0].to(dev)

    for _ in range(warmup_steps):
        _ = model(x0)
    _sync_if_cuda(dev)

    batch_times = []
    total_samples = 0

    for inputs, _ in data_loader:
        inputs = inputs.to(dev)

        _sync_if_cuda(dev)
        t0 = time.perf_counter()
        _ = model(inputs)
        _sync_if_cuda(dev)
        t1 = time.perf_counter()

        batch_times.append((t1 - t0) * 1000.0)
        total_samples += inputs.size(0)

    total_time_ms = float(np.sum(batch_times))
    avg_batch_ms = float(np.mean(batch_times))
    std_batch_ms = float(np.std(batch_times))
    avg_sample_ms = total_time_ms / max(total_samples, 1)
    throughput = total_samples / (total_time_ms / 1000.0)

    return {
        "num_batches": len(batch_times),
        "num_samples": total_samples,
        "total_time_ms": total_time_ms,
        "avg_batch_ms": avg_batch_ms,
        "std_batch_ms": std_batch_ms,
        "avg_sample_ms": avg_sample_ms,
        "throughput_sps": throughput,
    }


stats = benchmark_inference(model, test_loader, runtime_device, warmup_steps=20)

print("=" * 70)
print("Transformer Inference Benchmark (Test Loader)")
print("=" * 70)
print(f"Device            : {runtime_device}")
print(f"Batches           : {stats['num_batches']}")
print(f"Samples           : {stats['num_samples']}")
print(f"Total time        : {stats['total_time_ms']:.2f} ms")
print(f"Avg batch latency : {stats['avg_batch_ms']:.3f} ± {stats['std_batch_ms']:.3f} ms")
print(f"Avg sample latency: {stats['avg_sample_ms']:.4f} ms/sample")
print(f"Throughput        : {stats['throughput_sps']:.2f} samples/sec")

Transformer Inference Benchmark (Test Loader)
Device            : cuda:0
Batches           : 2
Samples           : 108
Total time        : 199.84 ms
Avg batch latency : 99.922 ± 18.772 ms
Avg sample latency: 1.8504 ms/sample
Throughput        : 540.42 samples/sec


In [11]:
# 실시간 환경(Batch Size 1)을 위한 전용 로더 생성
realtime_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# 기존의 벤치마크 함수를 그대로 사용하여 측정
stats_rt = benchmark_inference(model, realtime_loader, runtime_device, warmup_steps=50)

print("=" * 70)
print("실시간 응답 시간 측정 결과 (Batch Size = 1)")
print("=" * 70)
print(f"Device            : {runtime_device}")
print(f"순수 모델 응답 시간 : {stats_rt['avg_batch_ms']:.3f} ms") 
print(f"초당 처리 가능 횟수 : {stats_rt['throughput_sps']:.2f} FPS")
print("=" * 70)

실시간 응답 시간 측정 결과 (Batch Size = 1)
Device            : cuda:0
순수 모델 응답 시간 : 2.580 ms
초당 처리 가능 횟수 : 387.54 FPS


In [ ]:
# Train/Val Loss Curve (history list 우선, 없으면 노트북 출력 로그에서 자동 추출)
import json
import re
from pathlib import Path
import matplotlib.pyplot as plt

NOTEBOOK_PATH = Path('/home/binghin2/Myproject/Research/WESAD_classification/Training/Transformer/Transformer_64hz.ipynb')

def _extract_loss_from_text(text):
    pattern = re.compile(
        r"Epoch\s*\[(\d+)/(\d+)\].*?Train Loss:\s*([0-9]*\.?[0-9]+).*?Val Loss:\s*([0-9]*\.?[0-9]+)",
        flags=re.IGNORECASE,
    )
    matches = pattern.findall(text)
    if not matches:
        return [], [], []

    epochs = [int(m[0]) for m in matches]
    train_losses = [float(m[2]) for m in matches]
    val_losses = [float(m[3]) for m in matches]
    return epochs, train_losses, val_losses

def _extract_loss_from_notebook_outputs(nb_path):
    if not nb_path.exists():
        return [], [], []

    with nb_path.open('r', encoding='utf-8') as f:
        nb = json.load(f)

    all_text = []
    for cell in nb.get('cells', []):
        for output in cell.get('outputs', []):
            text = output.get('text', None)
            if isinstance(text, list):
                all_text.extend(text)
            elif isinstance(text, str):
                all_text.append(text)

    merged = ''.join(all_text)
    return _extract_loss_from_text(merged)

# 1) 런타임에 history 리스트가 이미 있으면 우선 사용
epochs = []
train_losses = []
val_losses = []

if 'train_losses' in globals() and 'val_losses' in globals():
    train_losses = list(train_losses)
    val_losses = list(val_losses)
    n = min(len(train_losses), len(val_losses))
    train_losses = train_losses[:n]
    val_losses = val_losses[:n]
    epochs = list(range(1, n + 1))
elif 'history' in globals() and isinstance(history, dict):
    t = history.get('train_loss', [])
    v = history.get('val_loss', [])
    n = min(len(t), len(v))
    train_losses = list(t[:n])
    val_losses = list(v[:n])
    epochs = list(range(1, n + 1))

# 2) history가 없으면 저장된 노트북 출력 로그에서 추출
if len(epochs) == 0:
    epochs, train_losses, val_losses = _extract_loss_from_notebook_outputs(NOTEBOOK_PATH)

if len(epochs) == 0:
    raise RuntimeError(
        'Train/Val loss 기록을 찾지 못했습니다. 학습 셀을 실행한 뒤 노트북 저장 후 다시 실행하세요.'
    )

plt.figure(figsize=(10, 5))
plt.plot(epochs, train_losses, marker='o', linewidth=2, label='Train Loss', color='#1f77b4')
plt.plot(epochs, val_losses, marker='o', linewidth=2, label='Val Loss', color='#d62728')
plt.title('Train vs Validation Loss Curve', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

best_idx = min(range(len(val_losses)), key=lambda i: val_losses[i])
print(f'Loaded epochs: {len(epochs)}')
print(f'Best Val Loss: {val_losses[best_idx]:.4f} (Epoch {epochs[best_idx]})')